# Robot session

Everything needed to bring the robot up, calibrate it, and hand coordinates to a
picking routine. Run the cells in order the first time; afterwards the
calibration sections can be re-run on their own.

The order in section 1 is not optional. `move_to_coordinates`, `move_relative`
and `get_position` all require both a run and a loaded pipette, so anything that
moves the robot fails until `create_run` and `load_pipette` have happened.

## 0. Setup

In [29]:
import os
# Notebooks live one level below the repository root.
os.environ["MICROPICK_ROOT"] = os.path.abspath("..")

import time
import numpy as np
import cv2

from opentrons_api import ot2_api

from micropick import paths
from micropick.config import store
from micropick.config.schema import CameraSpec, PipetteOffset
from micropick.hardware import labware
from micropick.hardware.camera import CameraManager
from micropick.hardware.protocols import xyz, goto_xy, move_to
from micropick.core.calibration.pixel_map import PixelMap, compare_degrees
from micropick.workflows.calibrate_camera import calibrate_camera
from micropick.workflows.calibrate_pipette import TipDetector, calibrate_pipette_offset
from micropick.workflows.jog import JogController, Limits, jog_in_window

paths.ensure_layout()
print(paths.describe())
print("\nprofiles:", store.list_profiles())

  root       ok       C:\Users\ivand\Desktop\micropick
  profiles   ok       C:\Users\ivand\Desktop\micropick\profiles
  labware    ok       C:\Users\ivand\Desktop\micropick\labware
  ml_models  ok       C:\Users\ivand\Desktop\micropick\ml_models
  outputs    ok       C:\Users\ivand\Desktop\micropick\outputs
  logs       ok       C:\Users\ivand\Desktop\micropick\logs

profiles: ['lab_main']


In [2]:
from micropick.hardware import labware
for d in labware.list_definitions().values():
    print(d)

greiner_1536_wellplate_12.6ul v1 (custom_beta), 1536 wells
wide_bore_200ul v1 (custom_beta), 96 wells
vwr_96_tiprack_200ul_xl v1 (custom_beta), 96 wells


### Profile

One profile per installation. Created once, then loaded on every later run.

In [ ]:
# PROFILE = "lab_main"

# profile = store.create_profile(PROFILE, camera_label="overview_cam",
#                                notes="OT-2, gantry camera", exist_ok=True)
# print(profile)
# print("positions:", sorted(profile.positions) or "none yet")

<Profile 'lab_main' (uncalibrated) at C:\Users\ivand\Desktop\micropick\profiles\lab_main>
positions: none yet


In [2]:
PROFILE = "lab_main"
profile = store.load_profile(PROFILE)
print(profile)
print("positions:", sorted(profile.positions) or "none yet")

<Profile 'lab_main' (uncalibrated) at C:\Users\ivand\Desktop\micropick\profiles\lab_main>
positions: none yet


### Cameras, first run only

Names must match what the operating system reports. Focus and exposure are
re-applied every time a camera is opened, which is what keeps a pixel map valid
across restarts: the map is only correct for the focus it was fitted at.

In [4]:
from micropick.hardware import devices
print(*devices.list_devices(), sep="\n")

[0] Arducam B0478 (USB3 48MP)
[1] Digital Microscope
[2] 20MP U3 Camera


In [5]:
profile.cameras = {
    "overview_cam": CameraSpec(
        device_name="20MP U3 Camera",
        resolutions=[[1280,720],
                    [1920,1080],
                    [2048,1536],
                    [2592,1944],
                    [3840,2160],
                    [4000,3000],
                    [4608,3456],
                    [5120,3840]],
        default_resolution=[2592, 1944], fps=30, fourcc="MJPG",
        controls={"auto_exposure": "manual"},
        notes="on the gantry, manual focus ring"),

    "underview_cam": CameraSpec(
        device_name="Arducam B0478 (USB3 48MP)",
        resolutions=[[1280,720],
                    [1920,1080],
                    [2000,1500],
                    [3840,2160],
                    [4000,3000],
                    [8000,6000]],
        default_resolution=[4000, 3000], fps=30, fourcc="MJPG",
        controls={"autofocus": 0, "focus": 920, "auto_exposure": "manual"},
        notes="tip calibration module, motorised focus"),
}
profile.save_cameras()
print(*profile.cameras.values(), sep="\n")

device_name='20MP U3 Camera' resolutions=[[1280, 720], [1920, 1080], [2048, 1536], [2592, 1944], [3840, 2160], [4000, 3000], [4608, 3456], [5120, 3840]] default_resolution=[2592, 1944] fps=30 fourcc='MJPG' controls={'auto_exposure': 'manual'} notes='on the gantry, manual focus ring'
device_name='Arducam B0478 (USB3 48MP)' resolutions=[[1280, 720], [1920, 1080], [2000, 1500], [3840, 2160], [4000, 3000], [8000, 6000]] default_resolution=[4000, 3000] fps=30 fourcc='MJPG' controls={'autofocus': 0.0, 'focus': 920.0, 'auto_exposure': 'manual'} notes='tip calibration module, motorised focus'


## 1. Robot

In [3]:
openapi = ot2_api.OpentronsAPI()
openapi.add_slot_offsets([5, 8, 9], (0, 0, 64.2))

In [4]:
openapi.toggle_lights()

<Response [200]>

In [4]:
# Use to restore labware and general run information after the notebook crashes
r = openapi.get_run_info()

Total number of runs: 20
Current run ID: c5ce637c-5ce6-4743-8eda-a996c88854d1
Current run status: idle


In [8]:
# Once after power on.
openapi.home_robot()

Request status:
<Response [200]>
{
  "message": "Homing robot."
}


<Response [200]>

In [9]:
openapi.create_run()
openapi.load_pipette()
print("run:", openapi.run_id, " pipette:", openapi.pipette_id)

Request status:
<Response [201]>
{
  "data": {
    "id": "c5ce637c-5ce6-4743-8eda-a996c88854d1",
    "ok": true,
    "createdAt": "2025-07-04T17:33:21.547902Z",
    "status": "idle",
    "current": true,
    "actions": [],
    "errors": [],
    "hasEverEnteredErrorRecovery": false,
    "pipettes": [],
    "modules": [],
    "labware": [],
    "liquids": [],
    "liquidClasses": [],
    "labwareOffsets": [],
    "runTimeParameters": [],
    "outputFileIds": []
  }
}
Request status:
<Response [201]>
{
  "data": {
    "id": "57f1aad9-f7c9-4e92-ab88-c8140484fb0d",
    "createdAt": "2025-07-04T17:33:22.281690Z",
    "commandType": "loadPipette",
    "key": "57f1aad9-f7c9-4e92-ab88-c8140484fb0d",
    "status": "succeeded",
    "params": {
      "pipetteName": "p300_single_gen2",
      "mount": "left"
    },
    "result": {
      "pipetteId": "05c40246-c85e-4a39-af0b-8e843a42b709"
    },
    "startedAt": "2025-07-04T17:33:22.284981Z",
    "completedAt": "2025-07-04T17:33:24.182580Z",
    "int

In [6]:
openapi.move_to_coordinates((100, 100, 150))

Request status:
<Response [201]>
{
  "data": {
    "id": "709df7b0-b2e7-4c3e-bf86-a435e4f07912",
    "createdAt": "2025-07-04T18:21:22.206955Z",
    "commandType": "moveToCoordinates",
    "key": "709df7b0-b2e7-4c3e-bf86-a435e4f07912",
    "status": "failed",
    "params": {
      "minimumZHeight": 20.0,
      "forceDirect": false,
      "pipetteId": "05c40246-c85e-4a39-af0b-8e843a42b709",
      "coordinates": {
        "x": 100.0,
        "y": 100.0,
        "z": 150.0
      }
    },
    "error": {
      "id": "13a61302-c25b-4bde-a215-aa2f377654fe",
      "createdAt": "2025-07-04T18:21:22.228142Z",
      "isDefined": false,
      "errorType": "FailedToPlanMoveError",
      "errorCode": "4000",
      "detail": "Destination out of bounds in the Z-axis",
      "errorInfo": {},
      "wrappedErrors": []
    },
    "startedAt": "2025-07-04T18:21:22.209715Z",
    "completedAt": "2025-07-04T18:21:22.228142Z",
    "intent": "setup",
    "notes": [
      {
        "noteKind": "debugErrorRecove

<Response [201]>

### Labware

Definitions live in `labware/` and are uploaded into the current run. This has
to happen again after every `create_run`. Names and namespaces come from the
files themselves.

In [10]:
for d in labware.list_definitions().values():
    print(d)

greiner_1536_wellplate_12.6ul v1 (custom_beta), 1536 wells
wide_bore_200ul v1 (custom_beta), 96 wells
vwr_96_tiprack_200ul_xl v1 (custom_beta), 96 wells


In [11]:
labware.ensure_definitions(openapi)

TIP_RACK = "vwr_96_tiprack_200ul_xl"
labware.load_labware(openapi, TIP_RACK, 10)

uploaded greiner_1536_wellplate_12.6ul v1 (custom_beta), 1536 wells
uploaded wide_bore_200ul v1 (custom_beta), 96 wells
uploaded vwr_96_tiprack_200ul_xl v1 (custom_beta), 96 wells


<Response [201]>

In [12]:
openapi.pick_up_tip(openapi.labware_dct["10"], "A4")

<Response [201]>

In [ ]:
openapi.move_labware(openapi.labware_dct['11'], 'offDeck')

## 2. Cameras

In [5]:
cams = CameraManager.from_profile(profile)
over_cam = cams.open("overview_cam")
print(over_cam)

overview_cam: [2] 20MP U3 Camera  2592x1944
  NOT applied: auto_exposure: asked 0.25, got 0
<BackgroundCamera 'overview_cam' 2592x1944 running, 0 frames>


In [ ]:
# The lower camera is only needed for tip calibration; open it there.
# cams.close("underview_cam")

In [18]:
def preview(camera, window="preview", size=(1348, 1011)):
    """Живой просмотр. Esc или q закрывает, s сохраняет кадр в outputs/images."""
    cv2.namedWindow(window, cv2.WINDOW_NORMAL)
    cv2.resizeWindow(window, *size)
    try:
        while True:
            ok, frame = camera.read()
            if not ok:
                continue
            vis = frame.copy()
            h, w = vis.shape[:2]
            cv2.drawMarker(vis, (w // 2, h // 2), (0, 0, 255), cv2.MARKER_CROSS, 60, 2)
            cv2.putText(vis, f"{w}x{h}  {camera.measure_fps(0.0) if False else ''}"
                             f"frames {camera.frame_count}", (20, 60),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.4, (0, 255, 0), 3)
            cv2.imshow(window, vis)
            key = cv2.waitKey(20) & 0xFF
            if key in (27, ord("q")):
                break
            if key == ord("s"):
                path = paths.images_dir() / f"{time.strftime('%H%M%S')}.png"
                cv2.imwrite(str(path), frame)
                print("saved", path)
    finally:
        cv2.destroyWindow(window)

# preview(over_cam)

### Jogging

Arrows or WASD move x and y, `q` and `e` move z, `+` and `-` change the step,
space saves a position, `u` undoes the last step, Enter finishes. The window
must have focus, so a stray keystroke in the notebook cannot drive the robot.

Limits are soft. Outside them the robot can always move back toward the working
area, only further out is refused.

In [6]:
LIMITS = Limits(x=(0, 380), y=(0, 350), z=(0.1, 150))

def jog(title="", camera=None, step=1.0):
    ctrl = JogController(openapi, limits=LIMITS, step=step)
    pos = jog_in_window(ctrl, camera or over_cam, title=title)
    print("stopped at", tuple(round(v, 2) for v in pos))
    return pos

In [17]:
jog()

stopped at (182.0, 162.5, 118.6)


(182.00446569789827, 162.50246866029056, 118.60000000000001)

## 3. Camera calibration

Fits lens distortion and the camera-to-robot relationship together from one
sweep of a static ArUco marker. No chessboard and no undistortion stage.

Redo it after any change to focus, zoom, camera height, or the height of the
plane the objects sit on.

In [19]:
openapi.toggle_lights()

<Response [200]>

In [7]:
MARKER_SIDE_MM = 6.8

aruco_dict = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_6X6_250)
params = cv2.aruco.DetectorParameters()
params.cornerRefinementMethod = cv2.aruco.CORNER_REFINE_SUBPIX
detector = cv2.aruco.ArucoDetector(aruco_dict, params)

Put the marker roughly in the centre of the frame and set Z to the height you
actually image the dish at. The sweep keeps whatever Z it starts from.

In [13]:
jog("centre the marker, set the working Z, then Enter")

stopped at (182.37, 162.67, 115.5)


(182.3739818153083, 162.66582295897254, 115.50000000000001)

In [9]:
pmap, report, sweep = calibrate_camera(
    openapi, over_cam, detector,
    marker_side_mm=MARKER_SIDE_MM, grid_n=7, degree=3,
    on_progress=lambda i, n: print(f"  {i}/{n}", end="\r"))

print("\n")
print(report)

measuring scale
  probe 1: 39 px, adjusting step to 13.44 mm
  scale: 13.438 mm moved the marker 513.3 px = 26.18 um/px
  field of view 67.9 x 50.9 mm, marker 264 px
planning: 7x7 poses, +/-29.0 x +/-21.0 mm, expected coverage 96 % x 96 % of the frame
sweeping 49 poses
  collected 49/49 poses, tracking id 1


degree 3, 49 poses, 196 points
  residual   mean    23.3  max    67.1 um
  held out   mean    25.2  max    98.6 um
  coverage   u [80, 2511]  v [23, 1875]  (94 % x 95 % of frame)
  scale      centre 25.98  edge 28.60 um/px (+10.1 %)
  track side 6.7508 mm


`track side` should come out near the printed marker size. The fit never uses
it, so agreement is independent evidence the sweep was good.

Degrees 1 and 2 should give identical numbers: radial distortion is cubic in
image coordinates, so a quadratic reduces exactly to an affine fit. A difference
between them would mean something other than lens distortion is in the data.

In [10]:
print(compare_degrees(sweep.track_px, sweep.gantry, sweep.image_size,
                      marker_side_mm=MARKER_SIDE_MM))

degree |    residual, um    |    held out, um    | track side
  1    |   205.0 /   946.9  |   214.2 /  1002.5 | 6.6900 mm
  2    |   204.0 /   839.1  |   227.3 /  1056.5 | 6.6900 mm
  3    |    23.3 /    67.1  |    25.2 /    98.6 | 6.7508 mm
  4    |    22.4 /    59.1  |    25.5 /   125.1 | 6.7508 mm
nominal track side 6.8000 mm


In [11]:
profile.calibration.pixel_map = pmap.to_config()
profile.save_calibration()
sweep.save(str(paths.fixtures_dir() / f"sweep_{time.strftime('%Y%m%d_%H%M')}.npz"))
print("saved")

saved


### Closed-loop check

Ask the map where the marker is, drive there, and see how far it lands from the
reference pixel. This is the only test that includes the robot.

Read the spread, not the absolute value. A consistent offset in one direction
with a small spread is the camera-to-tip constant and belongs to the pipette
offset; scatter is the map and the robot's repeatability.

In [14]:
def marker_centre_now():
    frame = over_cam.read_after(time.monotonic())
    corners, ids, _ = detector.detectMarkers(frame)
    if ids is None or len(corners) == 0:
        return None
    return corners[0].reshape(4, 2).mean(axis=0)

origin = xyz(openapi)
errors = []
for dx, dy in [(0, 0), (10, 7), (-12, -8), (18, -11), (-20, 12)]:
    openapi.move_to_coordinates((origin[0] + dx, origin[1] + dy, origin[2]-1),
                                min_z_height=1, verbose=False)
    time.sleep(0.4)
    g = xyz(openapi)[:2]                       # read next to the frame
    q = marker_centre_now()
    if q is None or not pmap.covers(*q):
        print(f"({dx:+3.0f},{dy:+3.0f}) not usable")
        continue

    target = pmap.to_robot(q[0], q[1], g)
    openapi.move_to_coordinates((target[0], target[1], origin[2]-1),
                                min_z_height=1, verbose=False)
    time.sleep(0.4)
    q2 = marker_centre_now()
    if q2 is None:
        continue
    err_px = q2 - np.array(pmap.config.ref)
    scale = float(np.mean(pmap.mm_per_px(*q2)))
    errors.append(err_px * scale)
    print(f"({dx:+3.0f},{dy:+3.0f})  residual {err_px[0]:+7.1f}, {err_px[1]:+7.1f} px"
          f"  = {np.linalg.norm(err_px) * scale * 1000:6.0f} um")

if errors:
    e = np.array(errors)
    print(f"\nbias   {e.mean(0)[0]*1000:+.0f}, {e.mean(0)[1]*1000:+.0f} um"
          f"   (constant, belongs to the pipette offset)")
    print(f"spread {np.linalg.norm(e - e.mean(0), axis=1).max()*1000:.0f} um max"
          f"   (this is the map plus robot repeatability)")

( +0, +0)  residual    -0.1,    +0.3 px  =      7 um
(+10, +7)  residual    -1.2,    +1.2 px  =     43 um
(-12, -8)  residual    +1.4,    -0.3 px  =     36 um
(+18,-11)  residual    -1.2,    -1.1 px  =     42 um
(-20,+12)  residual    +1.4,    +0.3 px  =     37 um

bias   +2, +2 um   (constant, belongs to the pipette offset)
spread 44 um max   (this is the map plus robot repeatability)


## 4. Pipette offset calibration

Redo this whenever a tip is picked up: every tip seats differently.

The upper camera locates the crosshair disc, the robot drives there using the
current offset, and the lower camera measures how far the tip actually is. The
gantry is parked a few millimetres to one side first, otherwise the tip covers
the crosshair and neither can be measured.

On a new installation the offset must be filled in roughly by hand first,
measured with a ruler. The routine drives to where it thinks the target is
before looking, so an offset that is wrong by tens of millimetres puts the tip
outside the lower camera's view and the run cannot recover.

In [15]:
from ultralytics import YOLO

tip_model = YOLO(str(paths.ml_models_dir() / profile.calibration.tip_target.model_file))
tip_detector = TipDetector(tip_model,
                           imgsz=profile.calibration.tip_target.imgsz,
                           conf=profile.calibration.tip_target.conf)
under_cam = cams.open("underview_cam")
print(under_cam)

underview_cam: [0] Arducam B0478 (USB3 48MP)  4000x3000
  NOT applied: auto_exposure: asked 0.25, got 0, autofocus: asked 0, got 1, focus: asked 920, got 1
<BackgroundCamera 'underview_cam' 4000x3000 running, 0 frames>


In [20]:
openapi.toggle_lights()

<Response [200]>

In [21]:
preview(under_cam)

First run only: fill in a rough offset measured with a ruler, and teach the
position of the calibration module.

In [22]:
if profile.calibration.pipette_offset is None:
    profile.calibration.pipette_offset = PipetteOffset(
        dx=16.0, dy=60.0, tip_type="vwr_200ul_xl", method="manual")
    profile.save_calibration(backup=False)
print(profile.calibration.pipette_offset)

dx=16.0 dy=60.0 tip_type='vwr_200ul_xl' measured_at=None method='manual' residual_mm=None n_samples=None spread_mm=None


In [23]:
# Teach where the crosshair disc is, once. Skip if it is already stored.
if "tip_calib" not in profile.positions:
    jog("bring the crosshair disc under the camera, then Enter")
    profile.remember("tip_calib", xyz(openapi))
print("tip_calib:", profile.where("tip_calib"))

stopped at (281.87, 161.76, 114.5)
tip_calib: (281.87353703094635, 161.75594045486756, 114.50000000000001)


In [24]:
target = profile.calibration.tip_target
openapi.move_to_coordinates(profile.where("tip_calib"),
                            min_z_height=target.module_height - 0.1, verbose=False)
time.sleep(0.5)

`manual_touch_up` runs after the automatic correction, with the lower camera
live. Nudge the tip onto the crosshair with a small step if the result is not
good enough, then press Enter. Whatever it moves is included, because the offset
is read from the final pose rather than from the commanded moves.

In [25]:
def touch_up(robot, camera, view):
    ctrl = JogController(robot, limits=LIMITS, step=0.05)
    jog_in_window(ctrl, camera, window="tip",
                  title="nudge the tip onto the crosshair, then Enter")

current = profile.calibration.pipette_offset
result = calibrate_pipette_offset(
    openapi, over_cam, under_cam, tip_detector, pmap,
    target=target,
    current_offset=(current.dx, current.dy),
    frames=7,
    tip_type=current.tip_type,
    manual_touch_up=touch_up)          # pass None to skip the manual step

print()
print(result)

upper camera: locating the crosshair
  crosshair at [      281.9      161.77] mm, spread 0.4 px over 7 frames
lower camera: measuring the tip
  residual [       15.2        -103] px = [     -2.462       0.363] mm, spread 0.5 px
  verification unavailable: no usable reading in 7 frames. Last problem: found 4 crosshairs, need the centre plus four neighbours; check lighting and focus
  manual touch-up moved [          0       -0.05] mm

offset          : dx  +16.538  dy  +60.313 mm
correction      :   -2.462,   +0.363 mm
upper camera    : 7 frames, spread 0.4 px
lower camera    : 7 frames, spread 0.5 px, 23.91 um/px
change from last: 0.622 mm
manual touch-up : +0.000, -0.050 mm


In [26]:
profile.calibration.pipette_offset = result.offset
profile.save_calibration(backup=False)
print("saved:", profile.calibration.pipette_offset)

saved: dx=16.53758221016068 dy=60.31261493423949 tip_type='vwr_200ul_xl' measured_at=datetime.datetime(2026, 8, 7, 22, 1, 44, 852981, tzinfo=datetime.timezone.utc) method='auto+manual' residual_mm=None n_samples=7 spread_mm=0.01142575209353833


## 5. Using the calibration

`pixel_to_robot` is the one function the picking code needs. The gantry pose has
to be read next to the frame the pixel came from: the pose is part of the
conversion, not a correction applied afterwards.

`mm_per_px` replaces the old global size ratio. Scale varies by several percent
across the frame, so a single number misreports objects near the edges.

In [27]:
profile = store.load_profile(PROFILE)
profile.require_calibration()
pmap = PixelMap.from_config(profile.pixel_map)
off = profile.calibration.pipette_offset
tip_offset = np.array([off.dx, off.dy])

problems = profile.pixel_map.check_camera(over_cam.resolution)
if problems:
    raise RuntimeError("the calibration does not match the camera: " + "; ".join(problems))

def pixel_to_robot(u, v, gantry_xy):
    """Robot coordinates that put the pipette tip on the pixel (u, v)."""
    if not pmap.covers(u, v):
        raise ValueError(f"pixel ({u:.0f}, {v:.0f}) is outside the calibrated area")
    return pmap.to_robot(u, v, gantry_xy) + tip_offset

def area_mm2(area_px, u, v):
    su, sv = pmap.mm_per_px(u, v)
    return area_px * su * sv

print("ready:", pmap.config.degree, "degree map,",
      f"holdout {pmap.config.holdout_mean_um:.1f} um,",
      f"offset ({off.dx:.2f}, {off.dy:.2f}) mm")

ready: 3 degree map, holdout 25.2 um, offset (16.54, 60.31) mm


In [ ]:
# Example: convert one detection.
g = xyz(openapi)[:2]
frame = over_cam.read_after(time.monotonic())
# u, v = ...detect something...
# tx, ty = pixel_to_robot(u, v, g)

In [37]:
profile.where("tip_calib")

(281.87353703094635, 161.75594045486756, 114.50000000000001)

In [40]:
def click_to_go(z=None, snap_px=60, conf=0.25, imgsz=2016, move=True):
    """
    Клик по кресту -> пипетка едет туда.

    Клик привязывается к ближайшей детекции, а не к сырым координатам курсора:
    попасть мышью в пиксель невозможно, а центр бокса модели субпиксельный.

    После переезда тот же крест находится заново, и его координаты считаются
    из новой позы гантри. Совпадение с прежними это и есть проверка карты по
    полю, для неё не нужно ничего измерять руками.

    Клавиши: d пересчитать детекции, r сбросить статистику, Esc выход.
    """
    win = "click to go"
    state = {"dets": [], "click": None, "frame": None, "gantry": None}
    history = []

    def on_mouse(event, x, y, flags, param):
        if event == cv2.EVENT_LBUTTONDOWN:
            state["click"] = (x, y)

        if event == cv2.EVENT_RBUTTONDOWN:
            openapi.move_to_coordinates(profile.where("tip_calib"))

    def detect_now():
        g = np.array(xyz(openapi)[:2])
        frame = over_cam.read_after(time.monotonic())
        res = tip_model.predict(source=frame[..., ::-1], conf=conf, imgsz=imgsz,
                                save=False, verbose=False)
        pts = []
        for r in res:
            for b in r.boxes:
                if tip_model.names[int(b.cls[0])] != "point":
                    continue
                x1, y1, x2, y2 = (float(v) for v in b.xyxy[0])
                p = np.array([(x1 + x2) / 2, (y1 + y2) / 2])
                if pmap.covers(*p):
                    pts.append((p, pmap.to_robot(p[0], p[1], g)))
        state["dets"], state["gantry"] = pts, g
        return pts

    print("детекция...")
    detect_now()
    print(f"найдено {len(state['dets'])} крестов")

    cv2.namedWindow(win, cv2.WINDOW_NORMAL)
    cv2.resizeWindow(win, 1348, 1011)
    cv2.setMouseCallback(win, on_mouse)
    try:
        while True:
            ok, frame = over_cam.read()
            if not ok:
                continue
            vis = frame.copy()
            h, w = vis.shape[:2]
            cv2.drawMarker(vis, tuple(np.int32(pmap.config.ref)), (0, 0, 255),
                           cv2.MARKER_CROSS, 60, 2)
            for p, world in state["dets"]:
                cv2.circle(vis, tuple(np.int32(p)), 14, (0, 255, 0), 2)
            cv2.putText(vis, f"{len(state['dets'])} crosses   click one   "
                             f"d=redetect  r=reset  Esc=quit", (20, 60),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 2)
            if history:
                e = np.array(history)
                cv2.putText(vis, f"map consistency: mean {e.mean()*1000:.0f} um, "
                                 f"max {e.max()*1000:.0f} um  (n={len(e)})",
                            (20, 110), cv2.FONT_HERSHEY_SIMPLEX, 1.2,
                            (0, 200, 255), 2)
            cv2.imshow(win, vis)

            key = cv2.waitKey(20) & 0xFF
            if key == 27:
                break
            if key == ord("d"):
                detect_now(); print(f"найдено {len(state['dets'])}")
            if key == ord("r"):
                history.clear()

            if state["click"] is None:
                continue
            cx, cy = state["click"]
            state["click"] = None
            if not state["dets"]:
                print("нет детекций, нажми d"); continue

            # привязка к ближайшему кресту в координатах отображаемого кадра
            # scale_x = frame.shape[1] / cv2.getWindowImageRect(win)[2]
            # scale_y = frame.shape[0] / cv2.getWindowImageRect(win)[3]
            # click_px = np.array([cx * scale_x, cy * scale_y])
            click_px = np.array([cx, cy])
            d = [np.linalg.norm(p - click_px) for p, _ in state["dets"]]
            k = int(np.argmin(d))
            # if d[k] > snap_px * max(scale_x, scale_y):
            #     print(f"мимо креста ({d[k]:.0f} px до ближайшего)"); continue
            if d[k] > snap_px:
                print(f"мимо креста ({d[k]:.0f} px до ближайшего)"); continue

            px, world = state["dets"][k]
            target = world + tip_offset
            print(f"\nкрест на пикселе ({px[0]:.0f}, {px[1]:.0f})")
            print(f"  координата креста : {world.round(3)}")
            print(f"  цель для пипетки  : {target.round(3)}")
            if not move:
                continue

            goto_xy(openapi, target[0], target[1])
            if z is not None:
                openapi.move_to_coordinates((target[0], target[1], z), min_z_height=1, verbose=False)
                # move_to(openapi, (target[0], target[1], z))

            # тот же крест заново, из новой позы
            after = detect_now()
            if after:
                worlds = np.array([wr for _, wr in after])
                j = int(np.argmin(np.linalg.norm(worlds - world, axis=1)))
                drift = float(np.linalg.norm(worlds[j] - world))
                history.append(drift)
                print(f"  тот же крест из новой позы: {worlds[j].round(3)}")
                print(f"  расхождение карты: {drift*1000:.0f} um")
    finally:
        cv2.destroyWindow(win)

    if history:
        e = np.array(history)
        print(f"\nсогласованность карты по {len(e)} переездам: "
              f"среднее {e.mean()*1000:.0f} мкм, максимум {e.max()*1000:.0f} мкм")
    return history



In [31]:
jog("Alignment")

stopped at (281.94, 161.68, 102.1)


(281.93698298292423, 161.67931078085462, 102.09999999999998)

In [41]:
drift = click_to_go(z = 67.0)

детекция...
найдено 5 крестов
Request status:
<Response [201]>
{
  "data": {
    "id": "6aae0df9-c7cc-421a-a7f8-d7285b17b612",
    "createdAt": "2025-07-04T19:56:06.338084Z",
    "commandType": "moveToCoordinates",
    "key": "6aae0df9-c7cc-421a-a7f8-d7285b17b612",
    "status": "succeeded",
    "params": {
      "minimumZHeight": 20.0,
      "forceDirect": false,
      "pipetteId": "05c40246-c85e-4a39-af0b-8e843a42b709",
      "coordinates": {
        "x": 281.87353703094635,
        "y": 161.75594045486756,
        "z": 114.50000000000001
      }
    },
    "result": {
      "position": {
        "x": 281.87353703094635,
        "y": 161.75594045486756,
        "z": 114.50000000000001
      }
    },
    "startedAt": "2025-07-04T19:56:06.340759Z",
    "completedAt": "2025-07-04T19:56:06.362704Z",
    "intent": "setup",
    "notes": []
  }
}

крест на пикселе (1315, 204)
  координата креста : [     282.28      181.98]
  цель для пипетки  : [     298.82      242.29]
Request status:
<Res

## 6. Bridge to the old picking code

The picking state machine has not been ported yet. To run it against this
calibration, replace two things in the old notebook and leave the rest alone.

```python
# was: X, Y, _ = tf_mtx @ (cX, cY, 1), then a gantry delta added
g = xyz(openapi)[:2]              # read immediately before the frame
X, Y = pixel_to_robot(cX, cY, g)

# was: area_mm = area_px * size_conversion_ratio
area_mm = area_mm2(area_px, cX, cY)
```

Everything else, the routines, the logger, the floater check, stays as it is.

## 7. Shutting down

In [ ]:
openapi.retract_axis("leftZ")
cams.close_all()